# baia-vision-poc — demo no Google Colab

Sobe o **front de teste** da POC com um **link público temporário** (via cloudflared, sem conta).
Roda com a CPU do Colab — e fica bem mais rápido se você ligar a GPU grátis.

**Como usar:**
1. (Opcional, recomendado) `Ambiente de execução` → `Alterar o tipo de ambiente` → **T4 GPU**.
2. Rode as células em ordem (▶️).
3. Na última célula, clique no link `https://....trycloudflare.com` e suba um vídeo.

> ⚠️ Use somente **footage royalty-free** (Pexels/Pixabay/Mixkit). Esta POC não é sistema de segurança de vida, não identifica pessoas e não lê placas.

In [ ]:
# 1) Baixa o código
![ -d ReconhecimentoPy ] || git clone -b main https://github.com/AliceCullen-html/ReconhecimentoPy.git
%cd /content/ReconhecimentoPy

In [ ]:
# 2) Instala dependências + cloudflared (túnel público)
#    O Colab já traz torch/opencv/numpy; instalamos o resto.
!pip install -q ultralytics flask gunicorn
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
print('✅ dependências prontas')

In [ ]:
# 3) Sobe o app, valida a saúde e abre o link público (só mostra a URL quando ela responde)
import os, subprocess, time, re, threading, urllib.request

# Colab tem CPU/GPU decente: modelo melhor e qualidade quase cheia.
os.environ.update({
    'MODEL_WEIGHTS': 'yolo11s.pt',  # maior/melhor que o yolo11n padrão
    'MODEL_CONF': '0.30',           # pega detecções um pouco mais fracas
    'MAX_UPLOAD_MB': '300',
    'PROC_MAX_WIDTH': '1280',       # 720p passa inteiro; limita só 4K
    'PROC_FRAME_STRIDE': '1',       # detecta todos os frames
})

APP_DIR = '/content/ReconhecimentoPy'

def reader(proc, buf):
    for line in proc.stdout:
        buf.append(line)

def get(url, timeout=4):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return r.status
    except Exception:
        return None

# --- servidor (gunicorn com threads: atende o polling enquanto processa) ---
server = subprocess.Popen(
    ['gunicorn', 'webapp.app:app', '--chdir', APP_DIR, '--pythonpath', APP_DIR,
     '--bind', '127.0.0.1:7860', '--workers', '1', '--worker-class', 'gthread',
     '--threads', '4', '--timeout', '1200'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
srv_log = []
threading.Thread(target=reader, args=(server, srv_log), daemon=True).start()

print('⏳ subindo o servidor…')
healthy = False
for _ in range(40):
    if server.poll() is not None:
        break
    if get('http://127.0.0.1:7860/') == 200:
        healthy = True
        break
    time.sleep(1)

if not healthy:
    print('❌ o servidor não respondeu. Log do servidor:')
    print(''.join(srv_log[-40:]) or '(sem saída)')
    raise SystemExit('Servidor não subiu — veja o log acima (e reinicie o ambiente se for erro de import).')
print('✅ servidor no ar (local)')

# --- túnel público ---
tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:7860', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url, t0 = None, time.time()
for line in tunnel.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
    if time.time() - t0 > 90:
        break
threading.Thread(target=reader, args=(tunnel, []), daemon=True).start()

# O túnel leva alguns segundos para registrar na borda; confirma antes de avisar.
public_ok = False
if url:
    print('🔗 URL criada, aguardando o túnel registrar…')
    for _ in range(25):
        if get(url, timeout=6) == 200:
            public_ok = True
            break
        time.sleep(2)

print('\n' + '=' * 66)
if url and public_ok:
    print('🚀 APP PÚBLICO (confirmado, pode abrir):')
    print('   ' + url)
elif url:
    print('🚀 URL:  ' + url)
    print('   O túnel ainda estava registrando — espere ~20s e atualize a página.')
else:
    print('Não capturei a URL — rode esta célula de novo.')
print('=' * 66)
print('Deixe esta célula rodando. O link morre quando o notebook fecha.')

**Para parar:** `Ambiente de execução` → `Interromper execução`.

**Se o link não abrir:** espere ~20s e atualize (o túnel demora a registrar); ou rode a célula 3 de novo.

**Detecção fraca?** Troque `MODEL_WEIGHTS` por `yolo11m.pt` (mais preciso, use GPU) na célula 3. E ajuste o polígono da zona em `config/config.yaml` para cobrir onde a operação acontece no seu vídeo.

**Dica de velocidade:** com a GPU T4 ligada, a inferência YOLO fica muito mais rápida. Para vídeos maiores em CPU, aumente `PROC_FRAME_STRIDE` (ex.: `2`).